# Follower POGEMA Baseline — Google Colab

Runs the **Follower** baseline (pure-Python, no FollowerLite) across the POGEMA `experiments/` folders, pulling code from **`tay805/mapf-deadlock`**.

**Before running:** set `Runtime > Change runtime type > GPU` (T4 is fine).

**Why a conda env?** Follower's stack is pinned to `torch 1.13.1 / sample-factory 2.1.1 / numpy<=1.23.1 / pandas<=1.4`, which only have wheels up to **Python 3.10**. Colab's native Python is 3.12, where those fail to build. So cell 1 installs conda (`condacolab`), and cell 4 builds a dedicated **`follower` env on Python 3.10**. The eval runs via `conda run -n follower …`, so the kernel's own Python (3.12) is irrelevant.

**Speed note:** GPU accelerates the policy net, but A* + env stepping stay CPU-bound and Colab free has ~2 vCPU. The win is offloading your machine, not raw speed.

## 1. Install Python 3.10 env (kernel will restart)

In [1]:
# Installs a Python 3.10 conda base. The kernel RESTARTS automatically at the end
# of this cell — that is normal. After it restarts, run cell 2 onward.
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:13
🔁 Restarting kernel...


## 2. Verify env + GPU (run AFTER the restart)

In [1]:
import condacolab; condacolab.check()   # confirms the conda base is active
!conda --version                        # needed to build the py3.10 env in cell 4
# The kernel's own Python can be 3.12 — we don't use it; the eval runs in the
# 'follower' conda env (py3.10) built in the deps cell.
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > GPU'

✨🍰✨ Everything looks OK!
conda 24.11.2
GPU 0: Tesla T4 (UUID: GPU-86635405-952a-db61-83ca-57525257a932)


## 3. Clone the project repo

Public repo, so no auth needed. Weights are committed (~59MB), so the clone brings them along.

To pull later changes without re-cloning: `!git -C /content/mapf-deadlock pull`

In [3]:
%cd /content
![ -d mapf-deadlock ] || git clone https://github.com/tay805/mapf-deadlock.git
%cd /content/mapf-deadlock
!ls -lh model/follower/checkpoint_p0/*.pth baseline_eval.py
# Mount Google Drive so generated RESULTS persist across sessions. Code stays in
# fast /content (re-cloning is cheap; checkpoints are in the repo); only results
# go to Drive via baseline_eval.py --out=RESULTS.
from google.colab import drive
drive.mount('/content/gdrive')
import os
DRIVE_ROOT='/content/gdrive/MyDrive/ColabNotebooks/NEP2022PROJECTS/MAPFv3'
RESULTS = f'{DRIVE_ROOT}/mapf-deadlock-results'
os.makedirs(RESULTS, exist_ok=True)
print('Results will be saved to:', RESULTS)

/content
/content/mapf-deadlock
-rw-r--r-- 1 root root 4.1K Jun  2 07:16 baseline_eval.py
-rw-r--r-- 1 root root  59M Jun  2 07:16 model/follower/checkpoint_p0/checkpoint_000061056_1000341504.pth
Mounted at /content/gdrive
Results will be saved to: /content/gdrive/MyDrive/ColabNotebooks/NEP2022PROJECTS/MAPFv3/mapf-deadlock-results


## 4. Build the Python 3.10 `follower` env + install deps

Creates a conda env on 3.10 and installs torch 1.13.1 + the eval's deps into it. Takes a few minutes. Must end with `OK | py 3.10.x | torch 1.13.1+cu117 | cuda True`.

In [4]:
# Follower's stack REQUIRES Python 3.10 (torch 1.13.1, sample-factory 2.1.1,
# numpy<=1.23.1, pandas<=1.4) — on Colab's native 3.12 these have no wheels and
# fail to build from source. So we build a dedicated conda env 'follower' on 3.10
# and run everything through `conda run -n follower`. The kernel's own Python
# doesn't matter. (Requires conda: cell 1 / condacolab must have run first.)
!conda --version || echo "!! conda missing — run cell 1 (condacolab) and let the kernel RESTART, then re-run this cell"
!apt-get -qq install -y build-essential >/dev/null
!conda create -y -n follower python=3.10 2>&1 | tail -2
# CUDA torch 1.13.1 (cu117 wheel exists for py3.10; runs on Colab T4).
!conda run -n follower pip install -q torch==1.13.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
# Everything else (wheels exist on 3.10). sample-factory pulls its own deps.
!conda run -n follower pip install -q --prefer-binary \
  sample-factory==2.1.1 pogema==1.3.0 pogema-toolbox==0.1.0 \
  "numpy>=1.21.5,<=1.23.1" "pandas<=1.4" matplotlib seaborn tabulate \
  pyyaml dask==2024.8.0 distributed==2024.8.0 loguru cppimport pybind11==2.13.1
# Verify INSIDE the env — must print py 3.10.x and torch 1.13.1+cu117.
!conda run -n follower python -c "import sys, torch, pogema, sample_factory, cppimport, pogema_toolbox, dask, loguru; print('OK | py', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

conda 24.11.2
#     $ conda deactivate

OK | py 3.10.20 | torch 1.13.1+cu117 | cuda True



## 5. Quick validation run (warehouse, 1 seed — a few minutes)

`baseline_eval.py` lives in the repo (Follower-only, no wandb, per-folder JSON, `--seeds=N`). Confirms the stack before the long run.

In [ ]:
# --no-capture-output streams logs live. First run pauses ~30-60s to compile the
# C++ A* planner. {RESULTS} is your Python var from cell 3 — IPython substitutes
# its value into the shell line (a bare RESULTS would be the literal string).
!cd /content/mapf-deadlock && conda run --no-capture-output -n follower python baseline_eval.py --seeds=1 05-warehouse --out "{RESULTS}"

## 6. Full baseline pass — all folders, full 10 seeds

Long-running. Keep the tab active so Colab doesn't disconnect. Folders are ordered so the cheap/high-value ones finish first and `01-random` (the big one, ~1600 episodes) runs last — so a disconnect during it still keeps the others (results save per-folder). For a faster first pass, append `--seeds=3`.

In [ ]:
# Full 10-seed pass, all folders, random last (so the valuable folders finish
# first; results save per-folder so a late disconnect keeps the rest).
# Add --seeds=3 for a faster first pass. {RESULTS} is interpolated from cell 3.
!cd /content/mapf-deadlock && conda run --no-capture-output -n follower \
  python baseline_eval.py 03-den520d 04-Paris_1 02-mazes 05-warehouse 01-random-20x20 --out "{RESULTS}"

In [ ]:
# --- Deadlock-metric pass (separate from the throughput run above) ---
# Adds deadlock_rate / recovery / unrecovered alongside throughput. Writes to a
# SEPARATE dir ({RESULTS}-deadlock) so it does NOT overwrite the throughput JSONs.
# den520d first (the headline map). Needs the --deadlock flag from baseline_eval.py
# -> make sure cell 3 cloned/pulled the latest code first.
!cd /content/mapf-deadlock && conda run --no-capture-output -n follower \
  python baseline_eval.py --deadlock \
  03-den520d 05-warehouse 04-Paris_1 02-mazes 01-random-20x20 \
  --out "{RESULTS}-deadlock"

In [ ]:
# --- Resolver pass: Follower + deadlock detector + PIBT waypoint-injection ---
# Writes throughput + deadlock + detector + resolver stats to a SEPARATE dir
# ({RESULTS}-resolve). Compare against the deadlock baseline ({RESULTS}-deadlock).
# Slower than baseline (PIBT runs on jam crops); den520d 256 is the slow case.
# den520d first (where the resolver helps most); random last.
!cd /content/mapf-deadlock && conda run --no-capture-output -n follower \
  python baseline_eval.py --resolve \
  03-den520d 05-warehouse 04-Paris_1 02-mazes 01-random-20x20 \
  --out "{RESULTS}-resolve"

## 7. Results

Results are already persisted to Google Drive at `MyDrive/mapf-deadlock-results/` (per-folder `*.json` + plots), so they survive a disconnect. The cell below just lists them. To also archive into the repo, copy them into `experiments/` and `git push`.

In [ ]:
# Results live on Drive (survive disconnects). List them:
!find "{RESULTS}" -name '*.json' -o -name '*.pdf' | sort
# Optional: copy into the repo and push so they're versioned on GitHub too.
# !cp -r "{RESULTS}"/* /content/mapf-deadlock/experiments/ \
#   && cd /content/mapf-deadlock && git add experiments && git commit -m "baseline results" && git push

## 8.  Head-to-head + bootstrap CIs (den520d-384)

 **S2** (statistics): the 4-way comparison (all-active /
congestion-routing / static metering / adaptive) on the over-saturated
`den520d-384` scenario, then **mean ± 95% bootstrap CI** + **paired bootstrap**
P(method > all-active).

**Per-seed & resumable.** The run cell does one `(condition, seed)` at a time and
writes a file *after every episode* (~5–10 min each), so you see progress
immediately and a disconnect never loses more than one episode — just re-run to
resume. `NSEEDS=10` is the reviewer headline requirement (S2).

*Pull first (gets the segfault fix + per-seed flag): `!git -C /content/mapf-deadlock pull`.*

In [ ]:
# --- Per-seed head-to-head on den520d-384 (S2). Runs one (cond, seed) at a
# time and saves after EACH episode -> incremental output, fully resumable,
# survives a session kill. ~5-10 min/episode; 4 conds x NSEEDS episodes. ---
import os, subprocess, time
RES = f"{RESULTS}-h2h"
NSEEDS = 10                                  #  headline requirement (S2)
conds = {
    'active':   [],                                       # all-active baseline
    'routing':  ['--route', '--deadlock'],                # congestion-routing proxy
    'static':   ['--meter=256', '--meter-mode=hide'],     # static density control
    'adaptive': ['--meter=384', '--meter-mode=adaptive'], # adaptive controller
}
for cond, flags in conds.items():
    for k in range(NSEEDS):
        out = f"{RES}/{cond}/seed{k}"
        done = f"{out}/08-val/Follower.json"
        if os.path.exists(done):
            print(f"{cond} seed{k}: done, skip", flush=True); continue
        print(f"=== {cond} seed{k} ===", flush=True); t0 = time.time()
        cmd = (['conda','run','--no-capture-output','-n','follower','python',
                'baseline_eval.py'] + flags + [f'--seed={k}', '08-val', f'--out={out}'])
        r = subprocess.run(cmd, cwd='/content/mapf-deadlock')
        dt = time.time() - t0
        ok = os.path.exists(done)
        print(f"    {'OK' if ok else f'FAILED rc={r.returncode}'} in {dt/60:.1f} min", flush=True)
        if not ok and r.returncode in (139, -11):
            print('    -> segfault: did you `git pull` for pogema_patch.py?', flush=True)
print('done -- run the CI cell next.')

In [ ]:
# --- bootstrap 95% CIs + paired test (reads per-seed files; run anytime) ---
import json, glob, numpy as np
RES = f"{RESULTS}-h2h"
def tps(c):                                  # gather every seed's throughput
    xs = []
    for fp in sorted(glob.glob(f"{RES}/{c}/seed*/08-val/Follower.json")):
        for r in json.load(open(fp)):
            xs.append(r['metrics']['avg_throughput'])
    return np.array(xs)
def ci(x, n=10000):
    bs = [np.mean(np.random.choice(x, len(x), replace=True)) for _ in range(n)]
    return x.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
base = tps('active')
print(f"den520d-384, {len(base)} seeds  (paired bootstrap vs all-active)")
print(f"{'method':<10}{'mean':>7}{'95% CI':>15}{'Δ%':>7}{'P(>base)':>9}")
for c in ['active','routing','static','adaptive']:
    x = tps(c)
    if len(x) == 0: print(f"{c:<10}  (no data yet)"); continue
    m,lo,hi = ci(x)
    n = min(len(x), len(base)); d = x[:n] - base[:n]       # paired by seed
    p = np.mean([np.mean(np.random.choice(d,len(d),replace=True))>0 for _ in range(10000)]) if n else float('nan')
    print(f"{c:<10}{m:>7.2f}  [{lo:.2f},{hi:.2f}]{(m/base.mean()-1)*100:>+6.0f}%{p:>9.2f}")

## 8b.  Second map: maze-640 head-to-head (10 seeds) — S2 + S4

Same 4-way comparison on the **maze** over-saturation map (`11-maze640`, 640 agents),
the localized-deadlock regime. Per-seed/resumable like §8. Closes the reviewer's
flagged 1-seed maze-adaptive number **and** gives a second map at full statistical
strength (S4). Static cap = 384 (near the maze peak). *(Pull latest first.)*

In [ ]:
# --- maze-640 per-seed head-to-head (S2/S4). Same structure as §8. ---
import os, subprocess, time
RESM = f"{RESULTS}-h2h-maze"
NSEEDS = 10
conds = {
    'active':   [],                                       # all-active baseline
    'routing':  ['--route', '--deadlock'],                # congestion-routing proxy
    'static':   ['--meter=384', '--meter-mode=hide'],     # static cap near maze peak
    'adaptive': ['--meter=640', '--meter-mode=adaptive'], # adaptive controller
}
for cond, flags in conds.items():
    for k in range(NSEEDS):
        out = f"{RESM}/{cond}/seed{k}"
        done = f"{out}/11-maze640/Follower.json"
        if os.path.exists(done):
            print(f"{cond} seed{k}: done, skip", flush=True); continue
        print(f"=== {cond} seed{k} ===", flush=True); t0 = time.time()
        cmd = (['conda','run','--no-capture-output','-n','follower','python',
                'baseline_eval.py'] + flags + [f'--seed={k}', '11-maze640', f'--out={out}'])
        r = subprocess.run(cmd, cwd='/content/mapf-deadlock')
        ok = os.path.exists(done)
        print(f"    {'OK' if ok else f'FAILED rc={r.returncode}'} in {(time.time()-t0)/60:.1f} min", flush=True)
        if not ok and r.returncode in (139, -11):
            print('    -> segfault: `git pull` for pogema_patch.py', flush=True)
print('done -- run the maze CI cell next.')

In [ ]:
# --- maze-640 bootstrap CIs + paired test (reads per-seed files) ---
import json, glob, numpy as np
RESM = f"{RESULTS}-h2h-maze"
def tps(c):
    xs = []
    for fp in sorted(glob.glob(f"{RESM}/{c}/seed*/11-maze640/Follower.json")):
        for r in json.load(open(fp)): xs.append(r['metrics']['avg_throughput'])
    return np.array(xs)
def ci(x, n=10000):
    bs = [np.mean(np.random.choice(x, len(x), replace=True)) for _ in range(n)]
    return x.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
base = tps('active')
print(f"maze-640, {len(base)} seeds  (paired bootstrap vs all-active)")
print(f"{'method':<10}{'mean':>7}{'95% CI':>15}{'Δ%':>7}{'P(>base)':>9}")
for c in ['active','routing','static','adaptive']:
    x = tps(c)
    if len(x)==0: print(f"{c:<10}  (no data yet)"); continue
    m,lo,hi = ci(x); n=min(len(x),len(base)); d=x[:n]-base[:n]
    p = np.mean([np.mean(np.random.choice(d,len(d),replace=True))>0 for _ in range(10000)]) if n else float('nan')
    print(f"{c:<10}{m:>7.2f}  [{lo:.2f},{hi:.2f}]{(m/base.mean()-1)*100:>+6.0f}%{p:>9.2f}")